# NB-00 · Partición Estratificada + Embeddings all-MiniLM-L6-v2
## Proyecto GATOBYTE — Amazon Electronics Sentiment Analysis

Este notebook es el cimiento de reproducibilidad de todo el proyecto. Hace exactamente dos cosas: define la partición de datos de forma única y permanente, y genera los embeddings semánticos que serán la representación de entrada para el modelo profundo. Ejecutarlo una sola vez y reutilizar sus artefactos garantiza que ningún experimento posterior pueda introducir contaminación entre splits ni variabilidad por re-muestreo accidental.

> ⚠️ Ejecutar este notebook **una sola vez**. Los archivos `.npy` y `.joblib` se reutilizan
> en todos los experimentos posteriores. Si ya existen, el notebook los detecta y salta la generación.


## 0 · Detección de entorno y rutas

El proyecto detecta automáticamente el entorno de ejecución. Esto es una decisión de ingeniería defensiva: elimina la necesidad de modificar rutas manualmente y hace el proyecto reproducible para cualquier integrante del equipo. El uso de pathlib.Path en lugar de strings de rutas es correcto porque maneja separadores de directorio de forma multiplataforma. La estructura de carpetas (data/, splits/, embeddings/, models/, reports/) sigue una convención estándar de proyectos ML que separa responsabilidades y facilita la auditoría de artefactos.

In [ ]:
import os, sys
from pathlib import Path

# ── Detectar si estamos en Colab o local ────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/ML')
    print('Entorno: Google Colab — Drive montado')
else:
    BASE_DIR = Path('.')
    print('Entorno: Local')

# ── Carpetas del proyecto ───────────────────────────────────────────────────
DIRS = {
    'data'      : BASE_DIR / 'data' / 'processed',
    'splits'    : BASE_DIR / 'splits',
    'embeddings': BASE_DIR / 'embeddings',
    'models'    : BASE_DIR / 'models',
    'reports'   : BASE_DIR / 'reports',
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print('\nCarpetas del proyecto:')
for k, v in DIRS.items():
    print(f'  {k:<12}: {v}')


## 1 · Instalación de dependencias

Se verifica si cada librería existe antes de instalar, evitando reinstalaciones redundantes en entornos donde ya está disponible. Esto es especialmente relevante en Colab, donde el kernel se reinicia entre sesiones. sentence-transformers es la única dependencia no estándar, y su instalación automática garantiza que el notebook sea autosuficiente.

In [ ]:
# Instalar sentence-transformers solo si no está disponible
import importlib, subprocess, sys

REQUIRED = {
    'sentence_transformers': 'sentence-transformers',
    'tqdm'                 : 'tqdm',
    'joblib'               : 'joblib',
}

for module, pkg in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    else:
        print(f'✓ {pkg} ya disponible')


Este fragmento de código se encarga de la gestión automatizada de dependencias en Python, asegurando que las librerías necesarias para el proyecto estén instaladas antes de continuar. El script define un diccionario (REQUIRED) con los módulos que se necesitan importar y sus respectivos nombres de instalación en pip (sentence-transformers, tqdm y joblib). Luego, recorre esta lista y utiliza la librería interna importlib para verificar si cada módulo ya está disponible en el entorno actual. Si el módulo no se encuentra (find_spec devuelve None), el código ejecuta un comando del sistema en segundo plano usando subprocess para instalar la librería mediante pip install -q (de forma silenciosa/oculta); si la librería ya estaba instalada, simplemente imprime un mensaje de confirmación (✓), evitando perder tiempo en reinstalaciones innecesarias.

## 2 · Imports y configuración global

***Decisiones clave: reproducibilidad y adherencia a la política de Transformers***.

Fijar SEED = 42 a nivel global es la práctica estándar para garantizar que la partición, el orden de inferencia y cualquier proceso estocástico sean idénticos en cada ejecución.

El parámetro MAX_SEQ_LENGTH = 150 es un ajuste liviano de tokenizer, no constituye fine-tuning. La separación de BATCH_SIZE_GPU = 512 vs BATCH_SIZE_CPU = 128 anticipa la variabilidad de hardware del equipo, alineada con la regla de oro del proyecto: el diseño no puede depender de hardware que no todos puedan ejecutar.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
import time
import gc
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

# ── Constantes ──────────────────────────────────────────────────────────────
SEED            = 42
EMB_MODEL_NAME  = 'sentence-transformers/all-MiniLM-L6-v2'
MAX_SEQ_LENGTH  = 150       # ajuste liviano de tokenizer (política §4.1 ✓)
BATCH_SIZE_GPU  = 512       # para GPU T4 (16 GB VRAM)
BATCH_SIZE_CPU  = 128       # para CPU (más conservador)
TARGET_COL      = 'sentiment'

np.random.seed(SEED)
print(f'SEED={SEED}  |  Modelo Transformer: {EMB_MODEL_NAME}')
print(f'MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}  |  (ajuste liviano, política §4.1 ✓)')


Este fragmento de código se encarga de la importación de librerías y la configuración inicial de constantes para un proyecto de procesamiento de lenguaje natural (NLP) y Machine Learning. En primer lugar, importa las herramientas esenciales para el manejo de datos y optimización (numpy, pandas, joblib, json, gc), desactiva las advertencias del sistema para mantener la consola limpia y carga módulos clave de scikit-learn (para dividir datos y codificar etiquetas) junto con tqdm para mostrar barras de progreso. En segundo lugar, establece e imprime las constantes críticas del pipeline: fija una semilla de aleatoriedad (SEED = 42) para garantizar que los resultados sean replicables, define el modelo de lenguaje que se utilizará (all-MiniLM-L6-v2 de Sentence-Transformers), limita la longitud máxima de los textos a 128 tokens para optimizar el rendimiento, configura tamaños de lote (batch sizes) diferenciados según si el procesamiento se hace en CPU o en una GPU específica (T4 de 16 GB), y define que la variable objetivo a predecir será el sentimiento (sentiment).

## 3 · Carga de datos

El patrón de búsqueda en múltiples rutas candidatas (lista CANDIDATOS) es robusto ante diferencias de directorio de trabajo entre entornos. Usar FileNotFoundError con mensaje descriptivo es preferible a fallar silenciosamente más adelante. Cargar en formato Parquet es correcto: es más eficiente en memoria y tiempo que CSV para datasets de este tamaño, y pandas lo soporta sin cuDF, preservando la compatibilidad con CPU.

In [ ]:
# ── Buscar el archivo parquet ───────────────────────────────────────────────
CANDIDATOS = [
    DIRS['data'] / 'sample_ml.parquet',
    BASE_DIR / 'data' / 'sample_ml.parquet',
    Path('../data/processed/sample_ml.parquet'),
    Path('data/processed/sample_ml.parquet'),
    Path('sample_ml.parquet'),
]

ruta_parquet = None
for ruta in CANDIDATOS:
    if ruta.exists():
        ruta_parquet = ruta
        print(f'✓ Datos encontrados: {ruta_parquet}')
        break

if ruta_parquet is None:
    raise FileNotFoundError(
        'No se encontró sample_ml.parquet. '
        'Asegúrate de ejecutar el script de sampleo primero.\n'
        f'Rutas buscadas: {[str(c) for c in CANDIDATOS]}'
    )

# ── Cargar con pandas (compatible CPU y Colab sin cuDF) ─────────────────────
t0 = time.time()
df = pd.read_parquet(ruta_parquet)
print(f'Dataset cargado en {time.time()-t0:.1f}s')
print(f'Shape: {df.shape}')
print(f'Columnas: {df.columns.tolist()}')
print(f'\nDistribución del target ({TARGET_COL}):')
print(df[TARGET_COL].value_counts(normalize=True).round(4).to_string())


Este fragmento de código busca, valida y carga de forma eficiente un conjunto de datos en formato Parquet para el proyecto. Para lograrlo, recorre una lista de rutas potenciales (CANDIDATOS) y selecciona la primera ubicación donde el archivo sample_ml.parquet realmente exista; si no lo encuentra en ninguna de ellas, detiene la ejecución arrojando un error controlado (FileNotFoundError). Una vez localizado, utiliza pandas para cargar el dataset en memoria de manera compatible con cualquier entorno (CPU o Colab), midiendo el tiempo de lectura e imprimiendo en pantalla las dimensiones de la tabla, sus columnas y la distribución porcentual de la variable objetivo (sentiment).

## 4 · Partición estratificada 70 / 15 / 15

**¿Por qué estratificada?** El dataset tiene un desbalance severo (~71.4% positivo, ~21.2% negativo, ~7.4% neutro). Sin estratificación, es probable que el split de validación o test carezca de muestras suficientes de la clase minoritaria (neutro), haciendo que las métricas no sean representativas. La estratificación preserva la distribución original en los tres subconjuntos.

**¿Por qué guardar índices y no subconjuntos completos?** Guardar solo los índices (idx_train.npy, etc.) y el LabelEncoder es una decisión de diseño correcta: ocupa mínimo espacio, permite reconstruir cualquier split desde el DataFrame original, y garantiza que todos los notebooks del proyecto usen exactamente los mismos registros. Compartir el LabelEncoder es fundamental para que la codificación negativo=0, neutro=1, positivo=2 sea consistente en todos los experimentos.


**¿Por qué el LabelEncoder se guarda junto con los splits?** Si cada notebook creara su propio encoder, las etiquetas numéricas podrían invertirse (el modelo A entrena con negativo=0 y el modelo B con negativo=2), haciendo que las comparaciones sean inválidas. Este es un error clásico en proyectos multi-notebook.


**Exclusión de columnas**: Se documenta explícitamente que rating, is_satisfied, average_rating y rating_number se excluyen por target leakage, ya que derivan de la misma calificación que se usa para construir el label de sentimiento. Su exclusión evita que los modelos aprendan una correlación trivial que no existiría en producción.

### Protocolo
- **Una sola vez**: la partición se crea aquí y se guarda como índices numpy.
- **Estratificada**: cada split mantiene la proporción original de clases (~74% pos / ~20% neg / ~6% neu).
- **Reproducible**: SEED=42 fijo. Cualquier experimento posterior carga estos índices.


In [ ]:
# ── Verificar si ya existe la partición ─────────────────────────────────────
SPLIT_FILES = {
    'idx_train': DIRS['splits'] / 'idx_train.npy',
    'idx_val'  : DIRS['splits'] / 'idx_val.npy',
    'idx_test' : DIRS['splits'] / 'idx_test.npy',
    'y_train'  : DIRS['splits'] / 'y_train.npy',
    'y_val'    : DIRS['splits'] / 'y_val.npy',
    'y_test'   : DIRS['splits'] / 'y_test.npy',
}
LE_PATH = DIRS['models'] / 'label_encoder.joblib'

splits_existen = all(f.exists() for f in SPLIT_FILES.values()) and LE_PATH.exists()

if splits_existen:
    print('✓ Partición ya existente — cargando...')
    idx_train = np.load(SPLIT_FILES['idx_train'])
    idx_val   = np.load(SPLIT_FILES['idx_val'])
    idx_test  = np.load(SPLIT_FILES['idx_test'])
    y_train   = np.load(SPLIT_FILES['y_train'])
    y_val     = np.load(SPLIT_FILES['y_val'])
    y_test    = np.load(SPLIT_FILES['y_test'])
    le        = joblib.load(LE_PATH)
    print(f'  Train: {len(idx_train):,}  |  Val: {len(idx_val):,}  |  Test: {len(idx_test):,}')
    print(f'  Clases: {le.classes_}')

else:
    print('Creando partición estratificada...')
    t0 = time.time()

    # LabelEncoder compartido por TODOS los experimentos del proyecto
    le    = LabelEncoder()
    y_enc = le.fit_transform(df[TARGET_COL].values)  # str → int
    idx   = np.arange(len(df))

    # 70% train / 30% temp
    idx_train, idx_temp, y_train, y_temp = train_test_split(
        idx, y_enc, test_size=0.30, stratify=y_enc, random_state=SEED
    )
    # 15% val / 15% test (50% del 30%)
    idx_val, idx_test, y_val, y_test = train_test_split(
        idx_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
    )

    # ── Guardar índices y labels ─────────────────────────────────────────────
    np.save(SPLIT_FILES['idx_train'], idx_train)
    np.save(SPLIT_FILES['idx_val'],   idx_val)
    np.save(SPLIT_FILES['idx_test'],  idx_test)
    np.save(SPLIT_FILES['y_train'],   y_train)
    np.save(SPLIT_FILES['y_val'],     y_val)
    np.save(SPLIT_FILES['y_test'],    y_test)
    joblib.dump(le, LE_PATH)

    print(f'✓ Partición creada en {time.time()-t0:.1f}s')
    print(f'  Train : {len(idx_train):>7,}  ({len(idx_train)/len(df)*100:.1f}%)')
    print(f'  Val   : {len(idx_val):>7,}  ({len(idx_val)/len(df)*100:.1f}%)')
    print(f'  Test  : {len(idx_test):>7,}  ({len(idx_test)/len(df)*100:.1f}%)')
    print(f'  Clases: {le.classes_}  (encodings: {list(range(len(le.classes_)))})')

    del idx, idx_temp, y_temp, y_enc
    gc.collect()


### 4.1 · Verificación de distribución por split

El gráfico de barras con porcentajes por clase en cada split sirve como prueba empírica de que la estratificación funcionó correctamente. No basta con asumir que train_test_split(..., stratify=y_enc) opera bien; la inspección visual confirma que la proporción de clases es virtualmente idéntica en train, val y test. Esto es especialmente importante para la clase neutro, que al ser ~6% podría desaparecer en splits pequeños si no se estratifica.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

PALETTE = {'positive':'#1D9E75', 'neutral':'#EF9F27', 'negative':'#E24B4A', 'bg':'#F8F7F4'}
CLASS_NAMES = le.classes_  # ['negative', 'neutral', 'positive']

splits_info = {
    'Train': y_train, 'Val': y_val, 'Test': y_test
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
fig.patch.set_facecolor(PALETTE['bg'])
fig.suptitle('Distribución de clases por split (estratificada)', fontsize=13, fontweight='bold')

for ax, (split_name, y_s) in zip(axes, splits_info.items()):
    unique, counts = np.unique(y_s, return_counts=True)
    clases  = [CLASS_NAMES[i] for i in unique]
    colores = [PALETTE[c] for c in clases]
    bars = ax.bar(clases, counts, color=colores, edgecolor='white', linewidth=0.8, width=0.6)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max()*0.02,
                f'{cnt:,}\n({cnt/len(y_s)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
    ax.set_title(f'{split_name}  ({len(y_s):,} muestras)', fontsize=11)
    ax.set_facecolor(PALETTE['bg'])
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x/1000)}K'))
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
ruta_fig = DIRS['reports'] / 'split_distribucion.png'
plt.savefig(ruta_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Gráfico guardado: {ruta_fig}')


Este fragmento de código se encarga de la visualización y auditoría del proceso de partición de datos, generando un gráfico de barras estético para confirmar que la estratificación se realizó correctamente. Utilizando matplotlib, el script configura un lienzo personalizado (con una paleta de colores fija para las categorías positive, neutral y negative) y crea tres subgráficos en paralelo para comparar los conjuntos de Train, Val y Test. Para cada subconjunto, calcula el conteo de muestras por clase, dibuja las barras con sus respectivos colores, añade etiquetas de texto que detallan el número exacto y el porcentaje que representa cada clase, y finalmente exporta la gráfica como una imagen de alta resolución en la carpeta de reportes antes de mostrarla en pantalla.

### 4.2 · Guardar metadata de la partición

Documentar la partición en un JSON estructurado (metadata_split.json) es una práctica de reproducibilidad que va más allá de los requisitos mínimos. Permite a cualquier evaluador verificar independientemente que los porcentajes son correctos, que la semilla es fija, y que las columnas excluidas tienen una razón documentada. Este archivo actúa como contrato formal entre este notebook y todos los que vienen después.


In [ ]:
metadata_split = {
    'seed'          : SEED,
    'total_filas'   : int(len(df)),
    'proporciones'  : {'train': 0.70, 'val': 0.15, 'test': 0.15},
    'tamaños'       : {
        'train': int(len(idx_train)),
        'val'  : int(len(idx_val)),
        'test' : int(len(idx_test)),
    },
    'clases'        : CLASS_NAMES.tolist(),
    'distribucion_train': {
        CLASS_NAMES[i]: {
            'n'  : int(c),
            'pct': round(float(c/len(y_train)*100), 2)
        }
        for i, c in zip(*np.unique(y_train, return_counts=True))
    },
    'distribucion_val': {
        CLASS_NAMES[i]: {
            'n'  : int(c),
            'pct': round(float(c/len(y_val)*100), 2)
        }
        for i, c in zip(*np.unique(y_val, return_counts=True))
    },
    'distribucion_test': {
        CLASS_NAMES[i]: {
            'n'  : int(c),
            'pct': round(float(c/len(y_test)*100), 2)
        }
        for i, c in zip(*np.unique(y_test, return_counts=True))
    },
    'columnas_excluidas': ['rating', 'is_satisfied', 'average_rating', 'rating_number'],
    'razon_exclusion'   : 'Target leakage documentado en FASE 3 del notebook original',
    'label_encoder_path': str(LE_PATH),
}

META_SPLIT_PATH = DIRS['splits'] / 'metadata_split.json'
with open(META_SPLIT_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata_split, f, indent=2, ensure_ascii=False)

print(f'✓ Metadata guardada: {META_SPLIT_PATH}')
print(json.dumps(metadata_split, indent=2))


Este fragmento de código se encarga de generar, guardar y exportar la metadata de las particiones de datos en un formato estructurado de tipo JSON. El script recopila toda la información clave del proceso de división en un diccionario (metadata_split), guardando parámetros técnicos como la semilla aleatoria, las proporciones y tamaños de cada conjunto (Train, Val, Test), los nombres de las clases y un desglose detallado (tanto en cantidad de muestras como en porcentaje) de la distribución de sentimientos en cada split. Adicionalmente, documenta decisiones críticas del proyecto, especificando qué columnas fueron excluidas para evitar la fuga de datos (target leakage), y concluye escribiendo esta información en un archivo .json de forma legible e imprimiendo el resultado en consola como registro de auditoría.

## 5 · Preparación de textos para embeddings

La decisión de combinar título y cuerpo de la reseña está justificada: el título suele contener el juicio global ("Excellent product", "Waste of money") mientras el texto provee contexto. Usar solo el cuerpo perdería señal discriminativa; usar solo el título sería insuficiente para casos ambiguos. El análisis de longitud en caracteres y tokens estimados valida que MAX_SEQ_LENGTH = 150 cubre adecuadamente el corpus sin truncar la mayoría de los textos.


In [ ]:
df['texto_completo'] = (
    df['title'].fillna('').astype(str).str.strip()
    + ' '
    + df['text'].fillna('').astype(str).str.strip()
).str.strip()

# Estadísticas de longitud
lens_chars = df['texto_completo'].str.len()
print('Estadísticas de longitud del texto completo (título + reseña):')
print(f'  Media   : {lens_chars.mean():.0f} chars')
print(f'  Mediana : {lens_chars.median():.0f} chars')
print(f'  P75     : {lens_chars.quantile(0.75):.0f} chars')
print(f'  P99     : {lens_chars.quantile(0.99):.0f} chars')
print(f'  Máximo  : {lens_chars.max():.0f} chars')

# Tokens estimados (aprox chars/4)
tokens_estimados = lens_chars / 4
pct_dentro_128 = (tokens_estimados <= 128).mean() * 100
print(f'\n  % reseñas dentro de 128 tokens (aprox): {pct_dentro_128:.1f}%')
print(f'  → max_seq_length=128 cubre el {pct_dentro_128:.0f}% del corpus ✓')


Este fragmento de código realiza la preparación del texto y el análisis de su longitud para validar si la configuración del modelo de lenguaje es adecuada. En primer lugar, concatena las columnas de título (title) y reseña (text) en una nueva variable llamada texto_completo, asegurando la limpieza del texto al rellenar valores nulos con espacios vacíos y eliminar espacios innecesarios en los extremos. En segundo lugar, calcula estadísticas descriptivas de los caracteres (media, mediana y percentiles) y realiza una estimación de tokens (asumiendo de forma aproximada que 4 caracteres equivalen a un token); esto sirve para evaluar qué porcentaje del corpus se mantiene por debajo del límite de 128 tokens, concluyendo con una confirmación visual de que dicha longitud es suficiente para cubrir la gran mayoría de las reseñas sin truncar información esencial.

## 6 · Detección de hardware y configuración del Transformer

El código hace explícito que all-MiniLM-L6-v2 se usa en modo inferencia pura: se llama .eval(), se desactivan todos los gradientes con param.requires_grad = False, y se verifica que los parámetros entrenables sean exactamente 0. Esto no es solo cumplimiento de la política del proyecto, es la práctica correcta: activar gradientes en inferencia desperdicia memoria y tiempo sin ningún beneficio.

In [ ]:
import torch

# ── Detectar GPU ─────────────────────────────────────────────────────────────
GPU_DISPONIBLE = torch.cuda.is_available()
DEVICE = 'cuda' if GPU_DISPONIBLE else 'cpu'

if GPU_DISPONIBLE:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH_SIZE_EMB = BATCH_SIZE_GPU
    print(f'✓ GPU detectada: {gpu_name}  ({vram_gb:.1f} GB VRAM)')
    print(f'  Batch size para embeddings: {BATCH_SIZE_EMB}')
else:
    BATCH_SIZE_EMB = BATCH_SIZE_CPU
    import os
    n_cpu = os.cpu_count()
    print(f'⚡ CPU-only  ({n_cpu} cores detectados)')
    print(f'  Batch size para embeddings: {BATCH_SIZE_EMB}')
    print('  Estrategias de velocidad para CPU activadas:')
    print('    - encode con show_progress_bar=True')
    print('    - normalize_embeddings=True (L2 in-place, sin paso extra)')
    print('    - Checkpoint cada split (guarda parcialmente si se interrumpe)')

# Tiempo estimado
n_total = len(df)
if GPU_DISPONIBLE:
    vel = 14000   # oraciones/seg en T4
else:
    import os
    vel = 500 * max(os.cpu_count() // 4, 1)   # ~500/seg base por hilo

mins_total = n_total / vel / 60
print(f'\nTiempo estimado para {n_total:,} textos:')
print(f'  ~{mins_total:.0f} minutos ({mins_total/60:.1f} horas)')


## 7 · Cargar `all-MiniLM-L6-v2` en modo inferencia

¿Por qué all-MiniLM-L6-v2? Es un modelo de 22M de parámetros que produce embeddings de 384 dimensiones, diseñado específicamente para similitud semántica de oraciones. Es óptimo para este proyecto porque: (a) corre en CPU en tiempo razonable, (b) sus embeddings L2-normalizados permiten usar distancia coseno directamente como producto punto, y (c) supera a TF-IDF en capturar matiz semántico (ej. "not bad" ≠ "bad"), lo cual es relevante para análisis de sentimiento.

In [ ]:
from sentence_transformers import SentenceTransformer

print(f'Cargando {EMB_MODEL_NAME}...')
t0 = time.time()

emb_model = SentenceTransformer(EMB_MODEL_NAME, device=DEVICE)
emb_model.max_seq_length = MAX_SEQ_LENGTH   # ajuste liviano de tokenizer (§4.1 ✓)

# NO se entrena ni fine-tunea
# Verificación explícita: ningún parámetro tiene requires_grad=True
emb_model.eval()
for param in emb_model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in emb_model.parameters())
print(f'✓ Modelo cargado en {time.time()-t0:.1f}s')
print(f'  Parámetros totales    : {total_params:,}')
print(f'  Parámetros entrenables: 0  (modo inferencia pura ✓)')
print(f'  Dimensión de embedding: {emb_model.get_sentence_embedding_dimension()}')
print(f'  Device                : {DEVICE}')
print(f'  max_seq_length        : {emb_model.max_seq_length} tokens')


Este fragmento de código detecta y configura el hardware (GPU o CPU) mediante PyTorch para optimizar la creación de embeddings. Si encuentra una GPU, extrae sus datos (nombre y VRAM) y asigna un tamaño de lote (batch size) alto; si no, se adapta a la CPU activando estrategias de ahorro de memoria y un lote más pequeño. Al final, calcula una estimación matemática del tiempo que tomará procesar todos los textos según la potencia del componente detectado.

## 8 · Generación de embeddings (con checkpoint y reanudación)

La función generar_embeddings_con_checkpoint implementa tolerancia a fallos mediante checkpoints intermedios cada 50.000 textos. Esto es una decisión de robustez crítica: si el proceso se interrumpe (timeout de Colab, corte de conexión), no se pierde todo el progreso. El diseño es correcto porque:

Verifica primero si el archivo final ya existe con el shape correcto (idempotencia).
Lee el checkpoint parcial y la metadata de progreso para reanudar exactamente donde se detuvo.
Aplica normalize_embeddings=True directamente en encode(), lo cual hace la normalización L2 in-place sin un paso adicional.
Valida la forma de la matriz final con un assert antes de guardar.
Limpia los archivos temporales de checkpoint al terminar.

La normalización L2 es una decisión técnica importante: hace que la similitud coseno entre embeddings sea equivalente al producto punto, simplificando los cálculos en clustering y recuperación semántica que se harán en notebooks posteriores.

In [ ]:
def generar_embeddings_con_checkpoint(
    textos       : list,
    split_name   : str,
    modelo       ,
    ruta_salida  : Path,
    batch_size   : int  = 256,
    checkpoint_cada: int = 50_000,
) -> np.ndarray:
    """
    Genera embeddings con soporte de checkpoint.
    Si se interrumpe, puede reanudar desde el último checkpoint.

    Parámetros
    ----------
    textos        : lista de strings
    split_name    : 'train' | 'val' | 'test'  (para logs)
    modelo        : SentenceTransformer en modo eval
    ruta_salida   : ruta final del .npy
    batch_size    : tamaño de lote (GPU=512, CPU=128)
    checkpoint_cada: guardar checkpoint cada N textos

    Retorna
    -------
    np.ndarray de shape (N, 384) float32
    """
    ruta_ckpt = ruta_salida.parent / f'{split_name}_checkpoint.npy'
    ruta_meta = ruta_salida.parent / f'{split_name}_checkpoint_meta.json'

    N = len(textos)

    # ── Verificar si ya existe el archivo final ───────────────────────────
    if ruta_salida.exists():
        emb = np.load(ruta_salida)
        if emb.shape[0] == N:
            print(f'  ✓ {split_name}: embeddings ya existentes ({emb.shape}) — saltando')
            return emb
        else:
            print(f'  ⚠ {split_name}: archivo existente con shape incorrecto '
                  f'({emb.shape[0]} ≠ {N}) — regenerando')

    # ── Verificar si hay checkpoint parcial ──────────────────────────────
    inicio = 0
    bloques_previos = []

    if ruta_ckpt.exists() and ruta_meta.exists():
        with open(ruta_meta) as f:
            meta_ckpt = json.load(f)
        inicio = meta_ckpt.get('procesados', 0)
        if inicio < N:
            bloques_previos = [np.load(ruta_ckpt)]
            print(f'  → Reanudando {split_name} desde índice {inicio:,} / {N:,}')
        else:
            inicio = 0

    # ── Generar embeddings por lotes ─────────────────────────────────────
    print(f'  Generando embeddings: {split_name}  ({N:,} textos, batch={batch_size})')
    t0      = time.time()
    bloques = bloques_previos.copy()

    for i in tqdm(range(inicio, N, checkpoint_cada),
                  desc=f'{split_name}', unit='bloque'):
        fin_bloque = min(i + checkpoint_cada, N)
        sub_textos = textos[i:fin_bloque]

        emb_bloque = modelo.encode(
            sub_textos,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,   # L2 norm in-place
        ).astype(np.float32)

        bloques.append(emb_bloque)

        # Guardar checkpoint parcial
        ckpt_actual = np.vstack(bloques)
        np.save(ruta_ckpt, ckpt_actual)
        with open(ruta_meta, 'w') as f:
            json.dump({'procesados': fin_bloque, 'total': N}, f)

    # ── Concatenar y guardar resultado final ─────────────────────────────
    embeddings = np.vstack(bloques)
    assert embeddings.shape == (N, modelo.get_sentence_embedding_dimension()), \
        f'Shape incorrecto: {embeddings.shape}'

    np.save(ruta_salida, embeddings)

    # Eliminar checkpoint temporal
    if ruta_ckpt.exists():  ruta_ckpt.unlink()
    if ruta_meta.exists():  ruta_meta.unlink()

    elapsed = time.time() - t0
    vel     = N / elapsed
    print(f'  ✓ {split_name}: {embeddings.shape}  |  {elapsed/60:.1f} min  '
          f'({vel:.0f} textos/seg)  →  {ruta_salida.name}')

    return embeddings


## 9 · Ejecutar generación de embeddings para los 3 splits

Los embeddings se generan por separado para cada split, extrayendo los textos mediante los índices guardados previamente. Esto previene un error sutil pero grave, si se generasen embeddings para todo el DataFrame y luego se filtraran, el orden de las filas debe coincidir exactamente con los índices, lo cual es frágil ante reindexaciones. Usar df['texto_completo'].iloc[idx_split].tolist() garantiza la correspondencia uno a uno entre cada embedding y su etiqueta y_train/y_val/y_test.

In [ ]:
SPLITS_CONFIG = [
    ('train', idx_train),
    ('val',   idx_val),
    ('test',  idx_test),
]

embeddings_dict  = {}
tiempos_generacion = {}

print('=' * 60)
print('Generación de embeddings all-MiniLM-L6-v2')
print(f'Total textos: {sum(len(i) for _,i in SPLITS_CONFIG):,}')
print('=' * 60)

t_total = time.time()

for split_name, idx_split in SPLITS_CONFIG:
    ruta_salida = DIRS['embeddings'] / f'{split_name}.npy'
    textos_split = df['texto_completo'].iloc[idx_split].tolist()
    t_split = time.time()

    emb = generar_embeddings_con_checkpoint(
        textos       = textos_split,
        split_name   = split_name,
        modelo       = emb_model,
        ruta_salida  = ruta_salida,
        batch_size   = BATCH_SIZE_EMB,
        checkpoint_cada = 50_000,
    )

    embeddings_dict[split_name]    = emb
    tiempos_generacion[split_name] = round(time.time() - t_split, 1)

print(f'\n✓ Todos los embeddings generados en {(time.time()-t_total)/60:.1f} min')
gc.collect()


Este fragmento de código ejecuta el bucle principal para procesar y guardar los embeddings de todos los conjuntos de datos (Train, Val y Test) de forma secuencial. A partir de una configuración predefinida, el script calcula el total de textos a procesar y recorre cada split para extraer las filas de texto correspondientes usando los índices guardados previamente en idx_split. Luego, invoca a la función de generación con soporte de checkpoints, pasándole los hiperparámetros de lote optimizados para el hardware disponible. Finalmente, almacena las matrices resultantes en un diccionario, registra de forma individual el tiempo que tomó procesar cada subconjunto, reporta el tiempo total acumulado de la ejecución y fuerza una limpieza en la memoria RAM mediante el recolector de basura (gc.collect()).

## 10 · Metadata de embeddings y verificación

Verificar que la norma L2 promedio sea ≈ 1.0 (con std ≈ 0) confirma que la normalización se aplicó correctamente. Si la norma media fuera, por ejemplo, 0.87, indicaría un error en el pipeline. Esta verificación convierte lo que podría ser un bug silencioso en un error detectado antes de entrenar cualquier modelo. La metadata también documenta que fine_tuning: false y modo_inferencia: true.


In [ ]:
# ── Tamaños en disco ─────────────────────────────────────────────────────
sizes_mb = {}
for split_name in ['train', 'val', 'test']:
    ruta = DIRS['embeddings'] / f'{split_name}.npy'
    sizes_mb[split_name] = round(ruta.stat().st_size / 1e6, 1)

# ── Estadísticas de los embeddings ───────────────────────────────────────
stats_emb = {}
for split_name, emb in embeddings_dict.items():
    stats_emb[split_name] = {
        'shape'       : list(emb.shape),
        'dtype'       : str(emb.dtype),
        'norm_media'  : round(float(np.linalg.norm(emb, axis=1).mean()), 4),
        'norm_std'    : round(float(np.linalg.norm(emb, axis=1).std()), 6),
        'valor_min'   : round(float(emb.min()), 6),
        'valor_max'   : round(float(emb.max()), 6),
        'size_mb'     : sizes_mb[split_name],
    }

# ── Guardar metadata ─────────────────────────────────────────────────────
metadata_emb = {
    'modelo'          : EMB_MODEL_NAME,
    'max_seq_length'  : MAX_SEQ_LENGTH,
    'batch_size_usado': BATCH_SIZE_EMB,
    'device'          : DEVICE,
    'normalize_L2'    : True,
    'dim_embedding'   : 384,
    'modo_inferencia' : True,
    'fine_tuning'     : False,
    'politica'        : 'Transformer en inferencia pura (§4.1 ✓)',
    'texto_entrada'   : 'title.strip() + " " + text.strip()',
    'tiempos_min'     : tiempos_generacion,
    'splits'          : stats_emb,
    'rutas'           : {
        k: str(DIRS['embeddings'] / f'{k}.npy') for k in ['train','val','test']
    },
}

META_EMB_PATH = DIRS['embeddings'] / 'metadata_embeddings.json'
with open(META_EMB_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata_emb, f, indent=2, ensure_ascii=False)

print('✓ Metadata de embeddings guardada')
print(json.dumps(metadata_emb, indent=2))


Este fragmento de código se encarga de la auditoría, control de calidad y documentación final del proceso de generación de embeddings.

Primero, calcula el peso físico en megabytes (size_mb) de los archivos .npy guardados en el disco para cada conjunto (Train, Val, Test). Luego, realiza un análisis estadístico de las matrices numéricas obtenidas, verificando sus dimensiones (shape), tipo de dato (dtype), valores máximos y mínimos, y validando mediante álgebra lineal que la norma L2 promedio sea exactamente 1 (lo que confirma que los vectores están correctamente normalizados). Finalmente, empaqueta todas estas métricas de calidad junto con la configuración técnica del modelo (nombre del Transformer, longitud de secuencia, tiempos de ejecución y rutas de guardado) en un diccionario global, exportándolo como un archivo estructurado metadata_embeddings.json para garantizar la trazabilidad y reproducibilidad del experimento.

## 11 · Verificación final — todos los archivos del proyecto

El checklist final de 13 archivos actúa como una prueba de integración del notebook completo. Si cualquier paso falló silenciosamente (error de escritura en disco, interrupción parcial), este bloque lo detecta antes de que los notebooks de modelado fallen con errores crípticos horas después. Mostrar el tamaño en KB de cada archivo permite detectar archivos corruptos o vacíos (un .npy de 0 bytes indicaría un fallo de guardado). El mensaje final → Continúa con NB-01_modelos_embeddings.ipynb hace explícito el orden de ejecución del proyecto.

In [ ]:
print('=' * 65)
print('  VERIFICACIÓN FINAL — Archivos generados')
print('=' * 65)

archivos_esperados = [
    # Splits
    (DIRS['splits'] / 'idx_train.npy', 'Índices train'),
    (DIRS['splits'] / 'idx_val.npy',   'Índices val'),
    (DIRS['splits'] / 'idx_test.npy',  'Índices test'),
    (DIRS['splits'] / 'y_train.npy',   'Labels train'),
    (DIRS['splits'] / 'y_val.npy',     'Labels val'),
    (DIRS['splits'] / 'y_test.npy',    'Labels test'),
    (DIRS['splits'] / 'metadata_split.json', 'Metadata split'),
    # Modelos compartidos
    (DIRS['models'] / 'label_encoder.joblib', 'LabelEncoder'),
    # Embeddings
    (DIRS['embeddings'] / 'train.npy',               'Embeddings train'),
    (DIRS['embeddings'] / 'val.npy',                 'Embeddings val'),
    (DIRS['embeddings'] / 'test.npy',                'Embeddings test'),
    (DIRS['embeddings'] / 'metadata_embeddings.json','Metadata embeddings'),
    # Reports
    (DIRS['reports'] / 'split_distribucion.png', 'Gráfico distribución splits'),
]

todos_ok = True
for ruta, descripcion in archivos_esperados:
    existe = ruta.exists()
    if existe:
        size_kb = round(ruta.stat().st_size / 1024, 1)
        print(f'  ✓  {descripcion:<35} ({size_kb:>8.1f} KB)  {ruta.name}')
    else:
        print(f'  ✗  {descripcion:<35} FALTANTE  →  {ruta}')
        todos_ok = False

print('=' * 65)
if todos_ok:
    print('  ✓ Todos los archivos generados correctamente')
    print('  → Continúa con NB-01_modelos_embeddings.ipynb')
else:
    print('  ⚠ Algunos archivos faltan. Revisa el log de errores.')


Este fragmento de código realiza la verificación final y el control de integridad de todo el pipeline de preparación de datos antes de avanzar a la etapa de modelado. El script define una lista de tuplas (archivos_esperados) que contiene las rutas y descripciones de todos los artefactos críticos generados en los pasos anteriores: índices, etiquetas (labels), codificadores, matrices de embeddings, gráficos de distribución y archivos JSON de metadatos. Posteriormente, recorre cada elemento para verificar su existencia física en el disco; si el archivo está presente, calcula su tamaño en kilobytes (KB) y muestra una confirmación visual exitosa (✓), mientras que si falta algún componente, arroja una alerta estructurada (✗) y marca el proceso como incompleto para asegurar que no se continúe el experimento con datos faltantes.